<a href="https://colab.research.google.com/github/Wayodeni/university/blob/main/ispm/4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Задание №4. Разработка спецификации требований к информационной системе (SRS)**

# **СПЕЦИФИКАЦИЯ ТРЕБОВАНИЙ К ПРОГРАММНОМУ ОБЕСПЕЧЕНИЮ**

## **1. ВВЕДЕНИЕ**

### **1.1. Цель документа**
Настоящий документ определяет функциональные и нефункциональные требования к разрабатываемой информационной системе — веб-сервису анализа ледовой обстановки на основе спутниковых данных.

Документ предназначен для:
- фиксации состава функций системы;
- определения ограничений и допущений;
- установления критериев качества;
- описания условий приемки системы.

Требования сформированы с учетом задач проекта:
- загрузка и отображение изображений;
- выделение области интереса;
- сегментация ледового покрова;
- классификация типов льда;
- расчет статистики по ROI.

### **1.2. Область применения**
Система предназначена для анализа ледовой обстановки по данным дистанционного зондирования Земли в веб-интерфейсе.

Основной сценарий использования включает:
- загрузку изображения пользователем;
- выбор области интереса;
- запуск сегментации и классификации;
- получение визуальных и количественных результатов по льду в выбранной области.

По результатам анализа аналогов система ориентирована на:
- **первичную аудиторию** — специалистов по ДЗЗ, криосфере и геопространственной аналитике;
- **вторичную аудиторию** — прикладных пользователей из логистики, мониторинга и государственных структур.

### **1.3. Определения, акронимы и сокращения**
- **ДЗЗ** — дистанционное зондирование Земли.
- **ROI** — Region of Interest, область интереса, выбранная пользователем для анализа.
- **Сегментация** — процесс попиксельного выделения объектов на изображении.
- **Классификация льда** — отнесение фрагментов ледового покрова к заданным типам или классам.
- **ML** — машинное обучение.
- **MVP** — минимально жизнеспособная версия продукта.
- **SAR** — радиолокационные данные с синтезированной апертурой.
- **UI** — пользовательский интерфейс.
- **API** — программный интерфейс взаимодействия между компонентами системы.

### **1.4. Пользовательские роли**

#### **Роль 1. Специалист-аналитик**
- **Описание:** основной пользователь системы, выполняющий предметный анализ ледовой обстановки.
- **Количество:** 5–20 пользователей на этапе пилотной эксплуатации.
- **Технический уровень:** средний или высокий.
- **Основные задачи:**
  - загрузка спутниковых данных;
  - выделение ROI;
  - запуск сегментации и классификации;
  - анализ статистики;
  - экспорт результатов.
- **Основание:** первичная аудитория включает специалистов по ледовой обстановке, ДЗЗ и геопространственным данным, которым важны точность, ROI-анализ и количественная оценка результатов.

#### **Роль 2. Прикладной пользователь**
- **Описание:** пользователь, ориентированный на результат анализа, а не на детали обработки данных.
- **Количество:** 10–50 пользователей при расширении доступа.
- **Технический уровень:** низкий или средний.
- **Основные задачи:**
  - просмотр результатов анализа;
  - интерпретация карт и статистики по выбранной территории;
  - использование системы как инструмента поддержки решений.
- **Основание:** вторичная аудитория включает логистику, судоходство, госструктуры и экологические организации, которым важны простота, наглядность и быстрый доступ к аналитике.

---

## **2. ФУНКЦИОНАЛЬНЫЕ ТРЕБОВАНИЯ**

### **2.1. Работа с областью интереса**

#### **ФТ-001: Выделение ROI пользователем**
- **Описание:** система должна предоставлять пользователю возможность вручную выделить область интереса на изображении прямоугольником или полигоном.
- **Входные данные:** координаты области, заданные пользователем через интерфейс.
- **Выходные данные:** геометрия ROI, сохраненная в сессии анализа.
- **Бизнес-правила:**
  - анализ должен выполняться только внутри выбранного ROI;
  - при отсутствии ROI система должна предложить пользователю выбрать область или подтвердить анализ всего изображения.
- **Приоритет:** критический.
- **Критерий проверки:** пользователь выделяет ROI, после чего система отображает его границы и использует только выбранную область при запуске обработки.
- **Обоснование:** ROI-first подход — одно из ключевых конкурентных преимуществ и основное отличие от data-centric аналогов.

#### **ФТ-002: Сброс и повторный выбор ROI**
- **Описание:** система должна позволять удалить ранее выбранную область интереса и создать новую без повторной загрузки изображения.
- **Входные данные:** команда пользователя на сброс ROI.
- **Выходные данные:** удаление предыдущей геометрии и возможность выбора новой области.
- **Бизнес-правила:**
  - при смене ROI предыдущие результаты сегментации и статистики должны помечаться как неактуальные.
- **Приоритет:** высокий.
- **Критерий проверки:** пользователь удаляет ROI и формирует новый, после чего система блокирует просмотр старой статистики до повторного запуска анализа.

### **2.3. Сегментация и классификация**

#### **ФТ-003: Сегментация ледового покрова**
- **Описание:** система должна выполнять сегментацию ледового покрова в пределах выбранного ROI и формировать маску льда.
- **Входные данные:** изображение и выбранная область ROI.
- **Выходные данные:** бинарная или многоклассовая маска сегментации.
- **Бизнес-правила:**
  - сегментация запускается только по явному действию пользователя;
  - при ошибке модели система должна зафиксировать событие в журнале и показать уведомление.
- **Приоритет:** критический.
- **Критерий проверки:** после запуска обработки пользователь получает визуальную маску сегментации в пределах ROI.
- **Основание:** сегментация ледового покрова обозначена как одна из обязательных функций системы.

#### **ФТ-004: Классификация типов льда**
- **Описание:** система должна классифицировать сегментированные пиксели льда по заданному набору классов.
- **Входные данные:** маска сегментации и исходное изображение или его признаки.
- **Выходные данные:** карта классификации льда с присвоенными классами.
- **Бизнес-правила:**
  - перечень классов должен быть фиксирован в настройках модели;
  - для пикселей с низкой уверенностью должна поддерживаться метка «неопределённый класс» при наличии такого режима.
- **Приоритет:** критический.
- **Критерий проверки:** по завершении обработки пользователь видит карту классов и легенду, содержащую все доступные категории.
- **Основание:** классификация типов льда входит в основной целевой функционал и отражает специализацию системы на ледовой аналитике.

#### **ФТ-005: Наложение результатов на исходное изображение**
- **Описание:** система должна отображать результаты сегментации и классификации поверх исходного изображения с возможностью включения и отключения слоев.
- **Входные данные:**
  - исходное изображение;
  - маска сегментации;
  - карта классификации.
- **Выходные данные:** комбинированное визуальное представление результатов.
- **Бизнес-правила:**
  - пользователь должен иметь возможность отдельно включать слой исходного изображения;
  - пользователь должен иметь возможность отдельно включать слой сегментации;
  - пользователь должен иметь возможность отдельно включать слой классификации.
- **Приоритет:** высокий.
- **Критерий проверки:** пользователь может переключать слои и визуально сравнивать результат модели с исходными данными.

### **2.4. Статистика и результаты анализа**

#### **ФТ-006: Расчет статистики по ROI**
- **Описание:** система должна автоматически рассчитывать количественные показатели по выбранной области интереса после завершения сегментации и классификации.
- **Входные данные:**
  - ROI;
  - маска сегментации;
  - карта классификации.
- **Выходные данные:**
  - площадь льда в ROI;
  - доля каждого класса льда;
  - количество пикселей по классам.
- **Бизнес-правила:**
  - статистика должна рассчитываться только на основе актуального ROI и последних результатов анализа.
- **Приоритет:** критический.
- **Критерий проверки:** после анализа в интерфейсе отображаются не менее трех метрик:
  - общая площадь льда;
  - процентное соотношение классов;
  - количество пикселей по классам.
- **Обоснование:** отсутствие интегрированной статистики по ROI было выделено как ключевая проблема аналогов, особенно Polar View и Copernicus, а автоматический расчет статистики — как одно из конкурентных преимуществ разрабатываемой системы.

#### **ФТ-007: Визуализация статистики**
- **Описание:** система должна представлять статистические результаты в табличном и графическом виде.
- **Входные данные:** рассчитанные показатели по ROI.
- **Выходные данные:**
  - таблица;
  - диаграмма распределения классов льда.
- **Бизнес-правила:**
  - таблица и график должны обновляться при каждом новом запуске анализа.
- **Приоритет:** высокий.
- **Критерий проверки:** после пересчета статистики пользователь видит обновленную таблицу и не менее одной диаграммы распределения классов.

#### **ФТ-008: Экспорт результатов анализа**
- **Описание:** система должна предоставлять пользователю возможность скачать результаты анализа в виде изображения с наложением слоев и таблицы статистики.
- **Входные данные:** завершенный сеанс анализа.
- **Выходные данные:**
  - файл изображения;
  - файл таблицы статистики в формате CSV или XLSX.
- **Бизнес-правила:**
  - экспорт доступен только после успешного завершения сегментации и расчета статистики.
- **Приоритет:** средний.
- **Критерий проверки:** пользователь скачивает файл статистики и изображение результата, а содержимое файлов соответствует данным, отображаемым в интерфейсе.

---

## **3. НЕФУНКЦИОНАЛЬНЫЕ ТРЕБОВАНИЯ**

### **3.1. Требования к производительности**

#### **НФТ-П-001: Время отображения изображения**
Система должна отображать изображение:
- не более чем за 5 секунд;
- при сетевом подключении не ниже 20 Мбит/с;
- при использовании серверной конфигурации проекта MVP.

#### **НФТ-П-002: Время обработки ROI**
Система должна выполнять:
- сегментацию;
- классификацию;
- расчет статистики

для ROI площадью до 1024×1024 пикселей не более чем за 30 секунд на целевой конфигурации:
- с одной GPU среднего класса;
- или эквивалентным CPU-режимом для демонстрационного стенда.

### **3.2. Требования к надежности**

#### **НФТ-Н-001: Устойчивость к ошибкам входных данных**
Система должна:
- корректно обрабатывать некорректные, поврежденные и неподдерживаемые файлы;
- не завершаться аварийно;
- выдавать пользователю понятное сообщение об ошибке.

### **3.3. Требования к безопасности**

#### **НФТ-Б-001: Безопасная передача данных**
При размещении в рабочем окружении система должна:
- использовать HTTPS для передачи пользовательских файлов и результатов анализа;
- не хранить загруженные данные дольше, чем это требуется для выполнения сеанса, если иное не включено явно.

### **3.4. Требования к удобству использования**

#### **НФТ-Ю-001: Простота базового сценария**
Пользователь без опыта работы с GIS-системами должен иметь возможность выполнить базовый сценарий:
- выбор ROI;
- запуск анализа;
- просмотр статистики

не более чем за 7 действий в интерфейсе.

#### **НФТ-Ю-002: Понятность интерфейса**
Все основные элементы управления должны иметь:
- русскоязычные подписи;
- всплывающие подсказки;
- визуальную группировку по этапам анализа:
  - выбор ROI;
  - обработка;
  - результаты.

### **3.5. Требования к совместимости**

#### **НФТ-С-001: Поддерживаемые браузеры**
Система должна корректно работать в актуальных версиях:
- Google Chrome;
- Mozilla Firefox;
- Microsoft Edge

без потери ключевого функционала.

#### **НФТ-С-002: Поддерживаемые форматы данных**
Система должна принимать к обработке как минимум:
- GeoTIFF;
- TIFF.

Для демонстрационных сценариев также должны поддерживаться:
- PNG;
- JPEG,

если они могут быть использованы без пространственной привязки.

### **3.6. Требования к масштабируемости**

#### **НФТ-М-001: Масштабируемость архитектуры**
Архитектура системы должна допускать:
- вынос модуля инференса в отдельный backend-сервис;
- сохранение пользовательского интерфейса без изменений при таком выносе.

#### **НФТ-М-002: Рост пользовательской нагрузки**
Система должна проектироваться так, чтобы на следующем этапе развития обеспечивать одновременную работу не менее 20 активных пользователей за счет разделения:
- UI;
- сервиса обработки;
- слоя хранения.

**Обоснование:** сравнительный анализ проекта фиксирует клиент-серверную архитектуру «Streamlit UI + ML backend + ROI-аналитика», а также переход от средней масштабируемости MVP к более высокой при выносе backend.

---

## **4. ТРЕБОВАНИЯ К ИНТЕРФЕЙСАМ**

### **4.1. Пользовательский интерфейс**
Система должна включать следующие основные экраны и функциональные зоны:
- панель выбора ROI;
- панель запуска сегментации и классификации;
- область отображения результатов;
- панель статистики и экспорта.

Интерфейс должен быть реализован как веб-приложение на Streamlit, что соответствует:
- выбранному стеку проекта;
- сценарию быстрого прототипирования;
- требованию низкого порога входа для пользователя.

### **4.2. Программный интерфейс (API)**
На этапе MVP внешний публичный API не является обязательным.

Внутренний интерфейс взаимодействия должен быть предусмотрен между:
- UI-модулем;
- модулем обработки изображений;
- модулем инференса.

При развитии системы допускается реализация отдельного REST API для:
- передачи ROI;
- запуска анализа;
- получения статистики;
- выгрузки результатов.

### **4.3. Интерфейсы с внешними системами**
Система может интегрироваться со следующими внешними компонентами:
- файловыми хранилищами для загрузки снимков;
- геопространственными библиотеками и сервисами обработки;
- модулями машинного обучения;
- внешними источниками спутниковых данных в будущих версиях.

---

## **5. ОГРАНИЧЕНИЯ И ДОПУЩЕНИЯ**

### **5.1. Ограничения**

#### **ОГР-001**
На этапе разработки система:
- не включает прогноз положения или движения льда;
- ограничивается анализом текущего состояния изображения.

#### **ОГР-002**
Точность сегментации и классификации зависит от:
- качества входных спутниковых данных;
- сезона;
- региона;
- применяемой модели.

#### **ОГР-003**
На этапе MVP система:
- ориентирована на ограниченную нагрузку;
- может потребовать выделения backend-сервисов при росте числа пользователей или объема вычислений.

### **5.2. Допущения**

#### **ДОП-001**
Предполагается, что пользователь обладает доступом к спутниковым данным, пригодным для анализа.

#### **ДОП-002**
Предполагается, что модель сегментации и классификации:
- заранее обучена;
- доступна системе в готовом виде.

#### **ДОП-003**
Предполагается, что серверное окружение обеспечивает достаточно ресурсов для обработки ROI типового размера.

---

## **6. КРИТЕРИИ ПРИЕМКИ**

### **6.1. Функциональная приемка**

#### **КП-Ф-001**
Система должна:
- позволять пользователю выделить ROI;
- позволять повторно изменить ROI без перезагрузки изображения.

#### **КП-Ф-002**
Система должна:
- выполнять сегментацию и классификацию по ROI;
- выводить результат на экран.

#### **КП-Ф-003**
Система должна:
- рассчитывать статистику по льду в ROI;
- отображать эту статистику в интерфейсе.

### **6.2. Нефункциональная приемка**

#### **КП-НФ-001**
Время:
- отображения изображения;
- выполнения базового анализа

должно соответствовать установленным порогам производительности.

#### **КП-НФ-002**
Система:
- не должна завершаться аварийно при ошибках загрузки файлов;
- должна корректно информировать пользователя об ошибке.

#### **КП-НФ-003**
Интерфейс должен:
- быть работоспособен в поддерживаемых браузерах;
- обеспечивать прохождение базового сценария без дополнительного обучения.

### **6.3. Документация**

#### **КП-Д-001**
Должно быть подготовлено руководство пользователя с описанием:
- выбора ROI;
- запуска анализа;
- чтения статистики;
- экспорта результатов.

#### **КП-Д-002**
Должно быть подготовлено техническое описание архитектуры системы, включающее:
- модули обработки изображений;
- модули сегментации;
- модули визуализации.

### **6.4. Тестирование**

#### **КП-Т-001**
Для каждого функционального требования должен быть выполнен:
- хотя бы один положительный тестовый сценарий;
- хотя бы один сценарий обработки ошибки.

#### **КП-Т-002**
Для сегментации и классификации должна быть выполнена:
- проверка качества на тестовом наборе данных.

#### **КП-Т-003**
Для производительности должен быть проведен как минимум:
- один замер времени отображения изображения;
- один замер времени выполнения анализа ROI.